In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.models import Swin_T_Weights, swin_t

from sklearn.metrics import (
    cohen_kappa_score, confusion_matrix, classification_report,
    f1_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

from tqdm import tqdm

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CELL 3 — Configuration
# Model: Swin Tiny 224
# ============================================================

# ── Paths ─────────────────────────────────────────────────────
CSV_PATH   = "data/raw/trainLabels.csv"
IMAGE_DIR  = "data/processed"
OUTPUT_DIR = "outputs_swin_weighted_ce_sampler"

# ── Fixed settings — same for all group members ───────────────
SEED            = 42
IMAGE_SIZE      = 224
BATCH_SIZE      = 64
NUM_EPOCHS      = 30
LR_BACKBONE     = 1e-5    # backbone learning rate
LR_HEAD         = 1e-4    # classifier head learning rate
WEIGHT_DECAY    = 0.01
VAL_SPLIT       = 0.15
NUM_WORKERS     = 8       
LABEL_SMOOTHING = 0.1
PATIENCE        = 7       # early stopping patience
ACCUMULATION    = 1       # gradient accumulation steps
WARMUP_EPOCHS = 2

# ── Class info ────────────────────────────────────────────────
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
NUM_CLASSES = 5

# ── Device ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Reproducibility ───────────────────────────────────────────
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/checkpoints", exist_ok=True)

LOG_PATH = f"{OUTPUT_DIR}/training_log.txt"

if "_LOG_FILE_HANDLE" in globals() and not _LOG_FILE_HANDLE.closed:
    _LOG_FILE_HANDLE.close()
if "_ORIGINAL_STDOUT" not in globals():
    _ORIGINAL_STDOUT = sys.stdout
if "_ORIGINAL_STDERR" not in globals():
    _ORIGINAL_STDERR = sys.stderr

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)

    def __getattr__(self, name):
        return getattr(self.streams[0], name)

_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)
sys.stdout = Tee(_ORIGINAL_STDOUT, _LOG_FILE_HANDLE)
sys.stderr = Tee(_ORIGINAL_STDERR, _LOG_FILE_HANDLE)

print("=" * 60)
print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Log file: {LOG_PATH}")
print("=" * 60)

print(f"Device: {device}")


In [ ]:
# ============================================================
# CELL 4 — Dataset visualisation: class distribution
# ============================================================

df = pd.read_csv(CSV_PATH)
print(f"Total images: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

# Class distribution
counts = df["level"].value_counts().sort_index()
print(f"\nClass distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  Class {i} ({name}): {counts.get(i, 0)} images "
          f"({counts.get(i, 0)/len(df)*100:.1f}%)")

# Plot
colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Dataset Class Distribution — Diabetic Retinopathy\n",
             fontsize=13, fontweight="bold")

# Bar chart
bars = axes[0].bar(CLASS_NAMES, counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Image Count per DR Grade")
axes[0].set_xlabel("DR Severity Grade")
axes[0].set_ylabel("Number of Images")
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 str(val), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(counts.values, labels=CLASS_NAMES, colors=colors,
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="black", linewidth=0.5))
axes[1].set_title("Class Proportion")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Class distribution plot saved!")

In [ ]:
# ============================================================
# CELL 5 — Sample images per DR grade
# ============================================================

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle("Sample Retinal Images — One Per DR Severity Grade\n",
             fontsize=13, fontweight="bold")

colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

for grade in range(5):
    # Get first image of this grade
    subset = df[df["level"] == grade]
    img_name = subset.iloc[0]["image"]

    for ext in [".jpeg", ".jpg", ".png"]:
        img_path = os.path.join(IMAGE_DIR, img_name + ext)
        if os.path.exists(img_path):
            break

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    axes[grade].imshow(image)
    axes[grade].set_title(f"Grade {grade}\n{CLASS_NAMES[grade]}",
                          fontsize=11, fontweight="bold",
                          color=colors[grade])
    axes[grade].axis("off")

    # Add coloured border
    for spine in axes[grade].spines.values():
        spine.set_edgecolor(colors[grade])
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sample images plot saved!")

In [ ]:
# ============================================================
# CELL 6 — Dataset class and transforms
# ============================================================

def get_transforms(image_size=224, mode="train"):
    if mode == "train":
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(45),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.1, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])


class DRDataset(Dataset):
    """
    Loads DR images using CSV labels file.
    Images are preprocessed with Ben Graham technique (already applied).
    CSV: image (filename without extension), level (0-4)
    """
    def __init__(self, csv_path, image_dir, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        self.samples   = []

        for _, row in self.df.iterrows():
            img_name = str(row["image"])
            label    = int(row["level"])
            for ext in [".jpeg", ".jpg", ".png"]:
                img_path = os.path.join(image_dir, img_name + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, label))
                    break

        print(f"Dataset loaded: {len(self.samples)} images")
        counts = Counter(label for _, label in self.samples)
        for i, name in enumerate(CLASS_NAMES):
            print(f"  Class {i} ({name}): {counts.get(i, 0)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_labels(self):
        return [label for _, label in self.samples]


class SubsetWithTransform(Dataset):
    """Applies different transform to val subset."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.dataset   = subset.dataset
        self.indices   = list(subset.indices)
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        base_idx = self.indices[idx]
        img_path, label = self.dataset.samples[base_idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, label


def get_sampler(labels):
    """
    Sqrt inverse frequency WeightedRandomSampler.
    Boosts minority classes without destroying majority class.
    Chosen after ablation study across 7 training runs.
    """
    counts         = np.bincount(labels)
    class_weights  = 1.0 / np.sqrt(counts.astype(float))
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


# Load dataset
print("Loading dataset...")
full_dataset = DRDataset(CSV_PATH, IMAGE_DIR,
                         transform=get_transforms(IMAGE_SIZE, "train"))

indices = np.arange(len(full_dataset))
labels = np.array(full_dataset.get_labels())

train_idx, val_idx = train_test_split(
    indices,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=labels,
    shuffle=True,
)

train_ds = Subset(full_dataset, train_idx.tolist())
val_ds = Subset(full_dataset, val_idx.tolist())

train_size = len(train_ds)
val_size = len(val_ds)


val_ds_clean = SubsetWithTransform(val_ds, get_transforms(IMAGE_SIZE, "val"))

train_labels = [full_dataset.samples[i][1] for i in train_ds.indices]
sampler = get_sampler(train_labels)
print("Using sqrt inverse frequency WeightedRandomSampler for training")


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    drop_last=True,
)

val_loader = DataLoader(
    val_ds_clean,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)


print(f"\nTrain: {train_size} images | Val: {val_size} images")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ============================================================
# CELL 7 — Model: Swin Tiny 224
# ============================================================

def build_swin_tiny_model(num_classes=5, dropout=0.3):
    """
    Swin Tiny — shifted-window Transformer for 224x224 image classification.

    Key innovation: shifted-window self-attention, which keeps computation
    efficient while allowing information to flow across local windows.

    Architecture:
    - Swin Tiny backbone (patch size 4, window size 7, image 224x224)
    - 4 hierarchical stages with shifted-window transformer blocks
    - Pretrained on ImageNet-1k

    For DR grading: pretrained backbone frozen initially, then
    fine-tuned with layer-wise learning rate decay.
    """
    weights = Swin_T_Weights.IMAGENET1K_V1
    model = swin_t(weights=weights)
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes)
    )
    return model


model = build_swin_tiny_model(NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: Swin Tiny patch4 window7 224")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
# Weighted CrossEntropyLoss is used in the next cell.
# Focal loss is intentionally disabled for this run.


In [ ]:
# ============================================================
# CELL 8 — Loss, optimiser, scheduler
# ============================================================

all_labels   = full_dataset.get_labels()
label_counts = [Counter(all_labels)[i] for i in range(NUM_CLASSES)]
print(f"Label counts: {label_counts}")

class_counts = torch.tensor(label_counts, dtype=torch.float32, device=device)
class_weights = 1.0 / torch.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
print(f"Weighted CE class weights: {[round(x, 4) for x in class_weights.detach().cpu().tolist()]}")

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING,
)
print(f"Using sqrt weighted CrossEntropyLoss + label smoothing={LABEL_SMOOTHING}")


# Layer-wise LR: backbone trains 10x slower than head
head_params     = list(model.head.parameters())
head_ids        = set(id(p) for p in head_params)
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params,     "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

warmup = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS,
)

cosine = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS - WARMUP_EPOCHS,
    eta_min=1e-6,
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup, cosine],
    milestones=[WARMUP_EPOCHS],
)
scaler = GradScaler("cuda", enabled=(device.type == "cuda"))

print("Loss, optimiser and scheduler ready!")

In [ ]:
# ============================================================
# CELL 9 — Training and validation functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (images, labels) in enumerate(tqdm(loader, desc="  Train", leave=False)):
        images = images.to(device, non_blocking=True) 
        labels = labels.to(device, non_blocking=True)

        with autocast("cuda"):
            outputs = model(images)
            loss    = criterion(outputs, labels) / ACCUMULATION

        scaler.scale(loss).backward()

        if (step + 1) % ACCUMULATION == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(device, non_blocking=True) 
        labels = labels.to(device, non_blocking=True)

        with autocast("cuda"):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        total_loss += loss.item()
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (total_loss / len(loader),
            np.array(all_preds),
            np.array(all_labels),
            np.array(all_probs))


print("Training functions defined!")

In [ ]:
# ============================================================
# CELL 10 — Training loop with early stopping
# ============================================================

best_qwk      = -1.0
no_improve    = 0
train_losses, val_losses, val_qwks = [], [], []
val_preds_final = val_labels_final = val_probs_final = None

print(f"Starting training — max {NUM_EPOCHS} epochs, early stopping patience={PATIENCE}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{NUM_EPOCHS}")

    train_loss, _, _ = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_preds, val_labels_ep, val_probs = validate(
        model, val_loader, criterion, device)
    scheduler.step()

    qwk      = cohen_kappa_score(val_labels_ep, val_preds, weights="quadratic")
    f1_macro = f1_score(val_labels_ep, val_preds, average="macro", zero_division=0)
    acc      = (val_preds == val_labels_ep).mean()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_qwks.append(qwk)

    print(f"  Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")
    print(f"  QWK: {qwk:.4f} | F1 macro: {f1_macro:.4f} | Acc: {acc:.4f}")

    if qwk > best_qwk:
        best_qwk            = qwk
        no_improve          = 0
        val_preds_final     = val_preds
        val_labels_final    = val_labels_ep
        val_probs_final     = val_probs
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "qwk": qwk,
        }, f"{OUTPUT_DIR}/checkpoints/best_swin_tiny.pth")
        print(f"  ✓ New best QWK {best_qwk:.4f} — saved!")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nTraining complete! Best QWK: {best_qwk:.4f}")

In [ ]:
# ============================================================
# CELL 11 — Training curves
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Swin Tiny Training Curves — Diabetic Retinopathy Grading\n",
             fontsize=13, fontweight="bold")

epochs_range = range(1, len(train_losses) + 1)

axes[0].plot(epochs_range, train_losses, "b-o", markersize=4, label="Train Loss")
axes[0].plot(epochs_range, val_losses,   "r-o", markersize=4, label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, val_qwks, "g-o", markersize=4, label="Val QWK")
axes[1].axhline(y=best_qwk, color="darkgreen", linestyle="--",
                label=f"Best QWK = {best_qwk:.4f}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("QWK")
axes[1].set_title("Validation Quadratic Weighted Kappa")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved!")

In [ ]:
# ============================================================
# CELL 12 — Confusion matrix
# ============================================================

cm      = confusion_matrix(val_labels_final, val_preds_final)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix — Swin Tiny DR Grading\n",
             fontsize=13, fontweight="bold")

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ["Confusion Matrix (Counts)", "Confusion Matrix (Normalised)"],
    ["d", ".2f"]
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_xlabel("Predicted Label", fontweight="bold")
    ax.set_ylabel("True Label", fontweight="bold")
    ax.set_title(title, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Print classification report
print("\n── Classification Report ──")
print(classification_report(val_labels_final, val_preds_final,
                             target_names=CLASS_NAMES, zero_division=0))
acc = (val_preds_final == val_labels_final).mean()
print(f"Overall Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"Best QWK         : {best_qwk:.4f}")
print(f"Macro F1         : {f1_score(val_labels_final, val_preds_final, average='macro', zero_division=0):.4f}")

In [ ]:
# ============================================================
# CELL 13 — ROC Curve + AUC
# ============================================================

y_true_bin = label_binarize(val_labels_final, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(10, 7))
colors  = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

auc_scores = []
for i, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], val_probs_final[:, i])
    auc_val     = auc(fpr, tpr)
    auc_scores.append(auc_val)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name} (AUC = {auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title(f"ROC Curves — Swin Tiny DR Grading\n"
             f"Mean AUC = {np.mean(auc_scores):.3f}",
             fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"ROC curves saved! Mean AUC: {np.mean(auc_scores):.4f}")

In [ ]:
# ============================================================
# CELL 14 — Per-class F1, Precision, Recall bar chart
# ============================================================

report = classification_report(val_labels_final, val_preds_final,
                                target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)

metrics_data = {
    "Precision": [report[c]["precision"] for c in CLASS_NAMES],
    "Recall":    [report[c]["recall"]    for c in CLASS_NAMES],
    "F1-Score":  [report[c]["f1-score"]  for c in CLASS_NAMES],
}

x      = np.arange(len(CLASS_NAMES))
width  = 0.25
colors = ["#3498db", "#e74c3c", "#2ecc71"]

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, vals) in enumerate(metrics_data.items()):
    bars = ax.bar(x + i*width, vals, width, label=metric,
                  color=colors[i], edgecolor="black", linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", fontsize=8, fontweight="bold")

ax.set_xlabel("DR Severity Grade", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Per-Class Precision, Recall & F1-Score — Swin Tiny\n",
             fontsize=13, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Per-class metrics chart saved!")

In [ ]:
# ============================================================
# CELL 15 — GradCAM Attention Heatmaps (Swin Tiny)
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Load best checkpoint
ckpt = torch.load(f"{OUTPUT_DIR}/checkpoints/best_swin_tiny.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Wrapper to make Swin Tiny output a plain tensor for GradCAM
class SwinTinyWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

wrapped_model = SwinTinyWrapper(model)
wrapped_model.eval()

# Target layer — last Swin Tiny transformer block
target_layers = [model.features[-1][-1].norm1]

# Reshape transform for Swin Tiny final feature tokens
def reshape_transform(tensor, height=7, width=7):
    if tensor.ndim == 4:
        return tensor.permute(0, 3, 1, 2)
    result = tensor.reshape(
        tensor.size(0), height, width, tensor.size(2)
    )
    return result.permute(0, 3, 1, 2)

cam = GradCAM(
    model=wrapped_model,
    target_layers=target_layers,
    reshape_transform=reshape_transform
)

# Get one image per class
val_transform = get_transforms(IMAGE_SIZE, "val")
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("GradCAM Attention Maps — Where Swin Tiny Looks for DR Features\n",
             fontsize=14, fontweight="bold")

colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

for grade in range(5):
    subset   = df[df["level"] == grade]
    img_name = subset.iloc[3]["image"]

    for ext in [".jpeg", ".jpg", ".png"]:
        img_path = os.path.join(IMAGE_DIR, img_name + ext)
        if os.path.exists(img_path):
            break

    # Load original image
    orig = cv2.imread(img_path)
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    orig_resized = cv2.resize(orig, (IMAGE_SIZE, IMAGE_SIZE))
    orig_float   = orig_resized.astype(np.float32) / 255.0

    # Prepare tensor
    input_tensor = val_transform(
        Image.fromarray(orig_resized)
    ).unsqueeze(0).to(device)

    # Generate CAM
    targets   = [ClassifierOutputTarget(grade)]
    grayscale = cam(input_tensor=input_tensor, targets=targets)
    cam_image = show_cam_on_image(orig_float, grayscale[0], use_rgb=True)

    # Row 1 — original
    axes[0, grade].imshow(orig_resized)
    axes[0, grade].set_title(f"Grade {grade}: {CLASS_NAMES[grade]}",
                              fontweight="bold", color=colors[grade])
    axes[0, grade].axis("off")

    # Row 2 — GradCAM overlay
    axes[1, grade].imshow(cam_image)
    axes[1, grade].set_title("GradCAM Overlay", fontsize=9)
    axes[1, grade].axis("off")

    # Row 3 — heatmap only
    axes[2, grade].imshow(grayscale[0], cmap="jet")
    axes[2, grade].set_title("Attention Heatmap", fontsize=9)
    axes[2, grade].axis("off")

# Row labels
for row, label in enumerate(["Original Image", "GradCAM Overlay", "Attention Heatmap"]):
    axes[row, 0].set_ylabel(label, fontsize=11, fontweight="bold",
                             rotation=90, labelpad=10)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/gradcam_heatmaps.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("GradCAM heatmaps saved!")

In [ ]:
# ============================================================
# CELL 16 — Final results summary
# ============================================================

acc      = (val_preds_final == val_labels_final).mean()
f1_macro = f1_score(val_labels_final, val_preds_final, average="macro", zero_division=0)
f1_per   = f1_score(val_labels_final, val_preds_final, average=None,   zero_division=0)

print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("Model: Swin Tiny patch4 window7 224")
print("Dataset: Processed Kaggle DR Dataset (35,126 images)")
print("=" * 60)
print(f"Best QWK (primary metric) : {best_qwk:.4f}")
print(f"Overall Accuracy          : {acc:.4f} ({acc*100:.2f}%)")
print(f"Macro F1                  : {f1_macro:.4f}")
print(f"Mean AUC                  : {np.mean(auc_scores):.4f}")
print("-" * 60)
print("Per-class F1 scores:")
for i, (name, score) in enumerate(zip(CLASS_NAMES, f1_per)):
    print(f"  Class {i} ({name:15s}): {score:.4f}")
print("-" * 60)
print("Imbalance handling strategies:")
print("  1. Sqrt inverse frequency WeightedRandomSampler")
print("  2. Sqrt inverse frequency class weights in CrossEntropy")
print(f"  3. Label smoothing = {LABEL_SMOOTHING}")
print("  4. Layer-wise learning rate decay")
print(f"  5. LR warmup {WARMUP_EPOCHS} epochs + cosine annealing")
print("=" * 60)
print(f"\nAll outputs saved to: {OUTPUT_DIR}/plots/")
print("Files saved:")
for f in os.listdir(f"{OUTPUT_DIR}/plots"):
    print(f"  {f}")